# AI Crypto Hedge Fund MVP

This notebook will contain the final self-contained technical solution structured according to Part 2 of the assignment.

## 0. Setup and Assumptions

Repository-relative paths and included data snapshots will be used for reproducibility.

In [ ]:
from ai_crypto_hedge_fund import project_root

ROOT = project_root()
ROOT

## 2.1 Baseline Strategy for a Single Cryptocurrency

The baseline experiment uses BTC/USDT and compares buy-and-hold with a transparent dual moving average crossover strategy. Signals are converted to executable positions with a one-period lag, transaction costs are charged from turnover, and metrics are calculated only on the out-of-sample test period. This gives the later AI-agent layer a simple reference policy: the agent can consume baseline momentum state, risk metrics, and drawdown state, then decide whether to keep, reduce, or block exposure.

In [ ]:
import json

baseline_metrics_path = ROOT / "reports" / "metrics" / "baseline_metrics.json"
if baseline_metrics_path.exists():
    baseline_metrics = json.loads(baseline_metrics_path.read_text())
    baseline_metrics["strategies"]
else:
    "Run scripts/run_baseline_strategy.py to generate baseline metrics."

## 2.2 Econometric, ML, and AI-Agent Strategy for a Single Cryptocurrency

The enhanced single-asset experiment keeps the BTC/USDT split from Part 2.1 and compares rolling econometric forecasts, a RandomForest direction classifier, and a deterministic agent layer. Features include lagged returns, rolling volatility, rolling means, momentum windows, and moving-average distance. The supervised target is the next one-minute return direction. Random-chance risk is checked with a 50% reference, majority-class reference, test direction accuracy, and ROC AUC where defined.

In [ ]:
model_metrics_path = ROOT / "reports" / "metrics" / "single_asset_model_comparison.json"
if model_metrics_path.exists():
    model_metrics = json.loads(model_metrics_path.read_text())
    model_metrics["comparison_table"]
else:
    "Run scripts/run_single_asset_models.py to generate model comparison metrics."

## 2.3 Static Portfolio Management on Historical Data

The static portfolio experiment expands the system from one BTC/USDT strategy to a six-asset liquid crypto universe: BTC, ETH, BNB, SOL, XRP, and ADA quoted against USDT. Portfolio weights are estimated on the training period only and then held fixed through the out-of-sample test period. The comparison includes equal weight, inverse-volatility weighting, and a constrained long-only max-Sharpe optimizer with a per-asset cap and inverse-volatility fallback.

The selected portfolio is the method with the highest out-of-sample Sharpe ratio. In live trading this result would need liquidity checks, fee-tier modeling, order sizing, slippage controls, custody limits, and periodic re-estimation; the MVP treats execution as simulated minute-close fills with transaction cost on the initial allocation.


In [ ]:
static_metrics_path = ROOT / "reports" / "metrics" / "static_portfolio_metrics.json"
if static_metrics_path.exists():
    static_metrics = json.loads(static_metrics_path.read_text())
    {
        "selected_method": static_metrics["selected_method"],
        "universe": static_metrics["universe"],
        "comparison": static_metrics["comparison_table"],
        "weights": static_metrics["weights"],
    }
else:
    "Run scripts/run_static_portfolio.py to generate static portfolio metrics."


## 2.4 Dynamic Portfolio Rebalancing

The dynamic rebalancing experiment compares the selected static max-Sharpe portfolio with two adaptive inverse-volatility policies: weekly scheduled rebalancing and threshold rebalancing when simulated passive holding drift exceeds 2 percentage points. Each new target weight vector is estimated only from trailing data available before the effective rebalance timestamp, then evaluated through the same transaction-cost-aware backtest layer as the static portfolio.

The strategy selection rule is highest out-of-sample Sharpe ratio. On the full test period, the dynamic inverse-volatility policies produced more diversified weights and explicit rebalance event logs, but they did not beat the static max-Sharpe reference after turnover and costs.


In [ ]:
rebalancing_metrics_path = ROOT / "reports" / "metrics" / "rebalancing_metrics.json"
if rebalancing_metrics_path.exists():
    rebalancing_metrics = json.loads(rebalancing_metrics_path.read_text())
    {
        "selected_strategy": rebalancing_metrics["selected_strategy"],
        "comparison": rebalancing_metrics["comparison_table"],
        "event_summary": rebalancing_metrics["event_summary"],
    }
else:
    "Run scripts/run_dynamic_rebalancing.py to generate dynamic rebalancing metrics."


## 2.5 Large-Universe Portfolio with Dynamic Rebalancing

The large-universe experiment uses the full 120-pair Binance spot snapshot and compares a broad equal-weight reference with two sparse weekly dynamic allocators. The sparse allocators rank assets from trailing momentum and volatility, select the top 20 active symbols, cap each asset at 8%, and scale gross exposure down when trailing portfolio volatility is above target. This avoids unstable dense covariance optimization across 100+ assets.

Operational monitoring should track data coverage, stale symbols, turnover, realized costs, volatility, drawdown, active-set stability, and drift between desired and actual balances. Fail-safes should include max order size, max rebalance turnover, stale-data blocks, exchange outage handling, order rejection handling, drawdown stops, and a manual kill switch.


In [ ]:
large_universe_metrics_path = ROOT / "reports" / "metrics" / "large_universe_metrics.json"
if large_universe_metrics_path.exists():
    large_universe_metrics = json.loads(large_universe_metrics_path.read_text())
    {
        "universe_size": large_universe_metrics["universe_size"],
        "selected_strategy": large_universe_metrics["selected_strategy"],
        "comparison": large_universe_metrics["comparison_table"],
        "data_files": large_universe_metrics["data_files"],
    }
else:
    "Run scripts/run_large_universe.py to generate large-universe metrics."
